In [ ]:
# Audit holdout data quality, validate application-time features, and compare its schema with training data.
import pandas as pd, numpy as np
h = pd.read_csv("../data/holdout_applications.csv", low_memory=False)
pat = lambda s: s.dropna().astype(str).str.replace(r"[A-Za-z]", "A", regex=True).str.replace(r"\d", "9", regex=True).value_counts().to_dict()
print("shape:", h.shape, "| duplicate ids:", h.id.duplicated().sum())
print("issue_d formats:", pat(h.issue_d)); print("earliest_cr_line formats:", pat(h.earliest_cr_line))
print("term:", h.term.unique().tolist(), "| application_type:", h.application_type.value_counts().to_dict())

def parse_date(s):
    out = pd.to_datetime(s, format="%b-%Y", errors="coerce")
    for f in ["%Y-%m-%d", "%m/%Y"]: out = out.fillna(pd.to_datetime(s, format=f, errors="coerce"))
    return out
issue, cr = parse_date(h.issue_d), parse_date(h.earliest_cr_line)
n = pd.to_numeric(h.term.astype(str).str.extract(r"(\d+)")[0]); r = pd.to_numeric(h.int_rate.astype(str).str.replace(r"[^0-9.]", "", regex=True)) / 1200
inst = h.loan_amnt * r / (1 - (1 + r) ** -n)
checks = {"issue_d unparsed": issue.isna().sum(), "issue_d range": f"{issue.min():%Y-%m} to {issue.max():%Y-%m}",
 "annual_inc < 1000": ((h.annual_inc > 0) & (h.annual_inc < 1000)).sum(), "loan_amnt > 40000": (h.loan_amnt > 40000).sum(),
 "installment mismatch > $1": ((inst - h.installment).abs() > 1).sum(), "fico low > high": (h.fico_range_low > h.fico_range_high).sum(),
 "revol_util < 0": (h.revol_util < 0).sum(), "bc_util < 0": (h.bc_util < 0).sum(), "dti > 100": (h.dti > 100).sum(),
 "state not 2 capital letters": (~h.addr_state.astype(str).str.fullmatch(r"[A-Z]{2}")).sum(), "credit line after issue": (cr > issue).sum(),
 "bureau_recovery_flag = 1": (h.bureau_recovery_flag == 1).sum() if "bureau_recovery_flag" in h else "column absent"}
print(pd.Series(checks).to_string())
print(h[["mths_since_last_delinq", "emp_length", "emp_title", "bc_util", "revol_util"]].isna().mean().round(3).to_dict())
train = pd.read_parquet("../data/clean_v2.parquet").columns
print("in training, not in holdout:", sorted(set(train) - set(h.columns))); print("in holdout, not in training:", sorted(set(h.columns) - set(train)))

In [ ]:
# Compare historical and holdout feature distributions to detect temporal drift and inconsistent data scales.
import sys; sys.path.append("..")
import pandas as pd
from src.contract_transform import parse_date
D = "../data/"
h = pd.read_csv(D + "holdout_applications.csv", low_memory=False)
app = pd.read_csv(D + "applications.csv", low_memory=False).drop_duplicates("id")
bur = pd.read_csv(D + "credit_bureau.csv", low_memory=False)
tr = app.merge(bur, on="id"); tr["year"] = parse_date(tr.issue_d).dt.year
cols = ["revol_util", "bc_util", "revol_bal", "total_rev_hi_lim", "fico_range_low", "acc_open_past_24mths", "num_tl_op_past_12m", "open_acc", "total_acc"]
q = lambda s: [round(float(x), 2) for x in s.quantile([.05, .5, .95])]
for c in cols:
    print(f"{c:22} 2014 {q(tr[tr.year == 2014][c])} | 2015 {q(tr[tr.year == 2015][c])} | 2016 {q(h[c])}")
impl = lambda d: d.revol_bal / d.total_rev_hi_lim.where(d.total_rev_hi_lim > 0) * 100
print("implied revol_util (balance/limit*100), median:", round(impl(tr[tr.year == 2015]).median(), 1), "(2015) vs", round(impl(h).median(), 1), "(2016)")

In [ ]:
# Detect fractional revolving-utilization values and update the data contract with a validated scaling rule.
import json
def frac_rule(d):
    u = pd.to_numeric(d.revol_util, errors="coerce").abs(); imp = d.revol_bal / d.total_rev_hi_lim.where(d.total_rev_hi_lim > 0) * 100
    return (u <= 1.5) & ((u * 100 - imp).abs() < (u - imp).abs())
print("rule triggers | training + 2015:", int(frac_rule(tr).sum()), "of", len(tr), "| holdout:", int(frac_rule(h).sum()), "of", len(h))
s = json.load(open("../contract/data_spec.json"))
for f in s["fields"]:
    if f["name"] == "revol_util":
        f["cleaning"] += "; if reported as a fraction of 1 (2016 holdout) it is multiplied by 100 when that matches revol_bal / total_rev_hi_lim * 100"
s["input_rules"].append("revol_util on a 0-1 scale is detected row by row against revol_bal / total_rev_hi_lim")
json.dump(s, open("../contract/data_spec.json", "w"), indent=1)

In [ ]:
# Validate historical fractional-utilization cases and investigate holdout records that do not meet the scaling rule.
import pandas as pd
D = "../data/"
h = pd.read_csv(D + "holdout_applications.csv", low_memory=False)
app = pd.read_csv(D + "applications.csv", low_memory=False).drop_duplicates("id")
bur = pd.read_csv(D + "credit_bureau.csv", low_memory=False)
tr = app.merge(bur, on="id")

def implied(d):
    lim = d.total_rev_hi_lim.where(d.total_rev_hi_lim > 0)
    return d.revol_bal / lim * 100

def frac_rule(d):
    u = pd.to_numeric(d.revol_util, errors="coerce").abs()
    imp = implied(d)
    return (u <= 1.5) & ((u * 100 - imp).abs() < (u - imp).abs())

t = tr[frac_rule(tr)].copy()
t["implied"] = implied(t).round(2)
cols = ["id", "issue_d", "revol_util", "revol_bal", "total_rev_hi_lim", "implied"]
print(t[cols].to_string(index=False))

m = h[~frac_rule(h)]
print(len(m), "holdout rows not triggered")
print("missing util:", int(m.revol_util.isna().sum()))
print("limit missing or zero:", int((m.total_rev_hi_lim.fillna(0) == 0).sum()))
print("util == 0:", int((m.revol_util == 0).sum()))
print("util > 1.5:", int((m.revol_util > 1.5).sum()))
print(m[["revol_util", "revol_bal", "total_rev_hi_lim"]].head(8).to_string(index=False))